<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.6-eval-merge-and-deploy/notebooks/exercises-7_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.6: Evaluation, Merging & Deployment

*Module 7 · Fine-Tuning LLMs LIVE CLASS*

> You fine-tuned a model. But is it actually better? How do you merge multiple LoRAs? How do you shrink a 14GB model to 4GB for local inference? This lesson covers the final mile: automated evaluation, model merging techniques, GGUF quantization, Ollama deployment, and production serving on Vertex AI.

`Benchmarks` · `A/B Testing` · `Model Merging` · `GGUF` · `Ollama` · `60 min`

---

## About this notebook

This notebook contains the **2 runnable code examples** from the published lesson `lesson-7.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Automated Evaluation — Benchmarks + LLM Judge — `01_evaluation.py`
2. Step 2 — Model Merging — Combine Multiple LoRAs — `02_merging.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers torch peft


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Automated Evaluation — Benchmarks + LLM Judge

Test your model on held-out data. Score with automated metrics AND LLM-as-judge.

**`01_evaluation.py`** · _Block 1 of 2_

FINE-TUNE EVALUATION — Automated + LLM-as-Judge


In [ ]:
# FINE-TUNE EVALUATION — Automated + LLM-as-Judge
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
judge = genai.GenerativeModel("gemini-2.0-flash")

class FineTuneEvaluator:
    def __init__(self, test_cases):
        self.cases = test_cases

    def evaluate(self, model_fn, label="model"):
        """Run test cases, score with LLM judge."""
        scores = {"accuracy":[], "helpfulness":[], "style":[]}
        for tc in self.cases:
            response = model_fn(tc["question"])
            for metric in scores:
                r = judge.generate_content(
                    f"Rate 0-1 the {metric} of this response.\n"
                    f"Q: {tc['question']}\nExpected: {tc.get('expected','')[:100]}\n"
                    f"Response: {response[:200]}\nReturn ONLY 0.0-1.0:")
                try: scores[metric].append(float(r.text.strip().split()[0]))
                except: scores[metric].append(0.5)
        avgs = {k: round(sum(v)/len(v),3) for k,v in scores.items()}
        avgs["label"] = label
        return avgs

    def compare(self, base_fn, finetuned_fn):
        """A/B test: base model vs fine-tuned."""
        base = self.evaluate(base_fn, "base")
        ft = self.evaluate(finetuned_fn, "fine-tuned")
        print(f"\n  {'Metric':15} {'Base':>8} {'Fine-tuned':>12} {'Delta':>8}")
        for m in ["accuracy","helpfulness","style"]:
            d = ft[m]-base[m]
            arrow = "▲" if d>0 else "▼"
            print(f"  {m:15} {base[m]:>8.3f} {ft[m]:>12.3f} {arrow}{abs(d):>6.3f}")
        return base, ft

# Demo
tests = [
    {"question":"What is the refund policy?", "expected":"Full refund 7 days, 50% 8-30, none after 30"},
    {"question":"How much does the GenAI course cost?", "expected":"14999 rupees with lifetime access"},
    {"question":"What are the prerequisites?", "expected":"Python basics and high school math"},
]

# Simulate base vs fine-tuned
def base_model(q): return "I'm not sure about Netsetos policies. Please check their website."
def finetuned(q): return "Full refund within 7 days. 50% from 8-30 days. None after 30. Contact [email protected]."

evaluator = FineTuneEvaluator(tests)
print("Fine-Tune Evaluation:\n")
evaluator.compare(base_model, finetuned)


> **What just happened?** Base model scored low on accuracy (generic “check the website” response). Fine-tuned model scored high (specific refund policy with details). The LLM judge scored three dimensions independently: accuracy, helpfulness, and style. A/B testing with automated scoring is how you PROVE fine-tuning worked. Never ship a model without comparing to the base.


### Step 2 · Model Merging — Combine Multiple LoRAs

Merge LoRA adapters into the base model. Or merge multiple fine-tuned models together.

**`02_merging.py`** · _Block 2 of 2_

MODEL MERGING — LoRA merge + multi-model merge


In [ ]:
# MODEL MERGING — LoRA merge + multi-model merge
print("Model Merging Techniques:\n")

# ── Technique 1: Merge LoRA into base model ──
merge_lora = """
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load LoRA adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    "./netsetos-lora",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
# Merge adapter weights into base model
merged = model.merge_and_unload()
merged.save_pretrained("./netsetos-merged")
tokenizer = AutoTokenizer.from_pretrained("./netsetos-lora")
tokenizer.save_pretrained("./netsetos-merged")
# Result: standalone model, no adapter dependency
"""

# ── Technique 2: Merge multiple models (mergekit) ──
mergekit_yaml = """
# mergekit config: SLERP merge of two fine-tuned models
# pip install mergekit
slices:
  - sources:
      - model: netsetos/support-v1
        layer_range: [0, 32]
      - model: netsetos/coding-v1
        layer_range: [0, 32]
merge_method: slerp
base_model: google/gemma-2-2b-it
parameters:
  t: 0.5    # 50/50 blend
dtype: bfloat16
# Run: mergekit-yaml config.yaml ./merged-output
"""

print("  1. LoRA Merge: adapter + base = standalone model")
print("     merge_and_unload() = permanent merge")
print("     Result: no adapter dependency, faster inference\n")

print("  2. Multi-Model Merge (mergekit):")
print("     SLERP: smooth interpolation between two models")
print("     TIES: resolve conflicts, keep important deltas")
print("     DARE: prune small deltas, keep large ones")
print("     Use case: merge support-bot + coding-bot = all-rounder\n")

print("  3. When to merge:")
print("     - Always merge LoRA before production deployment")
print("     - Multi-model merge for combining specialized skills")
print("     - Test merged model on eval suite before shipping")


> **What just happened?** GGUF is the universal format for local/CPU deployment. Q4_K_M uses mixed precision for the best size-quality tradeoff. Unsloth handles the entire pipeline in a single API call.


---

## Wrap-up

You've walked through all 2 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
